In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
from google.colab import files
files.upload()  # upload kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"akshaymina","key":"b2a00067a05a52a44d7ae2d5f5e59f8f"}'}

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
!pip install -q kaggle

In [4]:
DATASET = "xdxd003/ff-c23"

!kaggle datasets download -d $DATASET -p /content/dataset

Dataset URL: https://www.kaggle.com/datasets/xdxd003/ff-c23
License(s): other
100% 16.7G/16.7G [02:43<00:00, 109MB/s]



In [ ]:
#!unzip -q /content/dataset/*.zip -d /content/data

In [5]:
!du -sh /content/dataset

17G	/content/dataset


In [6]:
!git clone https://github.com/akshaymina/SFANet.git

Cloning into 'SFANet'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 34 (delta 9), reused 32 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 1.32 MiB | 27.09 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [ ]:
%cd SFANet

/content/SFANet


In [7]:
# --- EDIT THESE PATHS ---
DRIVE = "/content/drive/MyDrive"
ZIP_FFPP   = "/content/dataset/ff-c23.zip"      # your FF++ zip path
ZIP_CELEB  = f"{DRIVE}/celeb-df-v2.zip"           # your Celeb-DF zip path
ZIP_WILD   = f"{DRIVE}/wild-deepfake.zip"      # your WildDeepfake zip path

WORK = "/content/sfanet_data"   # working dir (fast local disk)
OUT  = f"{DRIVE}/sfanet_preprocessed"  # save final output to Drive

# Demo mode: set to 5 for quick test, None for full dataset
MAX_VIDEOS = 100
MAX_FRAMES = 10      # faces extracted per video
FRAME_STRIDE = 10    # process every Nth frame
COMPRESSION = "c23"  # FF++ quality: raw | c23 | c40

In [9]:
"""

from google.colab import drive
drive.mount("/content/drive")

!pip install -q opencv-python-headless retina-face facenet-pytorch albumentations torch torchvision

"""



import os, sys, shutil, zipfile, subprocess
from pathlib import Path

# Option A: full SFANet folder on Drive (includes preprocessing/)
SFANET_DRIVE = f"{DRIVE}/SFANet"
if Path(SFANET_DRIVE, "preprocessing").exists():
    if Path("/content/SFANet").exists():
        shutil.rmtree("/content/SFANet")
    shutil.copytree(SFANET_DRIVE, "/content/SFANet")
    print("Using SFANet from Drive (with preprocessing/)")
else:
    # Option B: clone GitHub (model only — upload preprocessing/ separately)
    if not Path("/content/SFANet").exists():
        !git clone https://github.com/akshaymina/SFANet.git /content/SFANet
    if not Path("/content/SFANet/preprocessing").exists():
        raise FileNotFoundError(
            "Upload your local SFANet/preprocessing/ folder to Drive at "
            f"{SFANET_DRIVE}/preprocessing/  OR copy it into /content/SFANet/"
        )

sys.path.insert(0, "/content/SFANet/preprocessing")
Path(WORK).mkdir(parents=True, exist_ok=True)
Path(OUT).mkdir(parents=True, exist_ok=True)
print("Ready.")

Ready.


In [12]:
## UN-ZIP Extract Files

def unzip_if_needed(zip_path, dest):
    dest = Path(dest)
    if dest.exists() and any(dest.iterdir()):
        print(f"Already extracted: {dest}")
        return dest
    print(f"Extracting {zip_path} -> {dest}")
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest)
    return dest

RAW_FFPP  = unzip_if_needed(ZIP_FFPP,  f"{WORK}/raw/ffpp")
#RAW_CELEB = unzip_if_needed(ZIP_CELEB, f"{WORK}/raw/celebdf")
#RAW_WILD  = unzip_if_needed(ZIP_WILD,  f"{WORK}/raw/wild")

# Print top-level folders so you can fix paths if layout differs
for name, p in [("FF++", RAW_FFPP)]:
    print(f"\n{name} contents:", os.listdir(p)[:15])

Extracting /content/dataset/ff-c23.zip -> /content/sfanet_data/raw/ffpp

FF++ contents: ['FaceForensics++_C23']


In [ ]:
!pip install --no-deps facenet-pytorch
!pip install retina-face

# Learn retina-face and MTCN



In [11]:
import sys

print("--- Checking System & Core Libraries ---")
print(f"Python version: {sys.version}")

# 1. Verify facenet-pytorch
try:
    from facenet_pytorch import MTCNN, InceptionResnetV1
    import torch

    # Initialize the model structure to verify weight configurations
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

    print("\n✅ facenet-pytorch: SUCCESS")
    print(f"   - Running on device: {device}")
except Exception as e:
    print(f"\n❌ facenet-pytorch: FAILED\n   Error: {e}")

# 2. Verify retina-face
try:
    from retinaface import RetinaFace
    from retinaface.commons import distance

    # Initialize the model blueprint to ensure imports work seamlessly
    model = RetinaFace.build_model()

    print("\n✅ retina-face: SUCCESS")
    print("   - Core model structure built successfully.")
except Exception as e:
    print(f"\n❌ retina-face: FAILED\n   Error: {e}")


--- Checking System & Core Libraries ---
Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


  0%|          | 0.00/107M [00:00<?, ?B/s]


✅ facenet-pytorch: SUCCESS
   - Running on device: cuda

❌ retina-face: FAILED
   Error: cannot import name 'distance' from 'retinaface.commons' (/usr/local/lib/python3.13/dist-packages/retinaface/commons/__init__.py)


In [ ]:
# Delete processed data first
#!rm -rf /content/sfanet_data/processed/*

In [13]:
from pathlib import Path
import subprocess
import sys

# Install facenet_pytorch if not already installed
#!pip install facenet-pytorch

#Chat Gpt generated

FFPP_RAW = "/content/sfanet_data/raw/ffpp/FaceForensics++_C23"

print("FF++ raw root:", FFPP_RAW)

if not Path(FFPP_RAW).exists():
    raise FileNotFoundError(
        f"Dataset root not found: {FFPP_RAW}"
    )

print("\nRaw dataset folders:")
for p in sorted(Path(FFPP_RAW).iterdir()):
    if p.is_dir():
        print(" -", p.name)


FFPP_OUT = f"{WORK}/processed/ffpp"

cmd = [
    sys.executable,
    "/content/SFANet/preprocessing/preprocess_ffpp.py",

    "--raw_root",
    FFPP_RAW,

    "--out_root",
    FFPP_OUT,

    "--compression",
    COMPRESSION,

    "--frame_stride",
    str(FRAME_STRIDE),

    "--max_frames",
    str(MAX_FRAMES),
]

if MAX_VIDEOS:
    cmd += [
        "--max_videos",
        str(MAX_VIDEOS)
    ]


print("\nRunning:")
print(" ".join(cmd))
print()

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

# Optionally re-raise the error if you want the cell to fail, but after seeing the output
result.check_returncode()

FF++ raw root: /content/sfanet_data/raw/ffpp/FaceForensics++_C23

Raw dataset folders:
 - DeepFakeDetection
 - Deepfakes
 - Face2Face
 - FaceShifter
 - FaceSwap
 - NeuralTextures
 - csv
 - original

Running:
/usr/bin/python3 /content/SFANet/preprocessing/preprocess_ffpp.py --raw_root /content/sfanet_data/raw/ffpp/FaceForensics++_C23 --out_root /content/sfanet_data/processed/ffpp --compression c23 --frame_stride 10 --max_frames 10 --max_videos 100

STDOUT:

SFANet FaceForensics++ Preprocessing
Raw dataset : /content/sfanet_data/raw/ffpp/FaceForensics++_C23
Output      : /content/sfanet_data/processed/ffpp
Compression : c23
Frame stride: 10
Max frames  : 10
Max videos  : 100

Using RetinaFace for face detection.

Raw video inventory:
  original       : 1000
  Deepfakes      : 1000
  Face2Face      : 1000
  FaceSwap       : 1000
  NeuralTextures : 1000


Found 500 videos to process.

Discarded cross-split fake videos: 272
Leakage check PASSED
Processing 228 retained videos...
[1/228] 000.

In [ ]:
!/usr/bin/python3 -c "from facenet_pytorch import MTCNN; print('OK')"

OK


In [ ]:
# ------------------------------------------------------------
# Verify the generated dataset
# ------------------------------------------------------------

verify_cmd = [
    sys.executable,
    "/content/SFANet/preprocessing/verify_datasets.py",

    "--dataset",
    "ffpp",

    "--root",
    FFPP_OUT,

    "--compression",
    COMPRESSION,
]

print("\nRunning dataset verification...\n")

result = subprocess.run(
    verify_cmd,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

# Optionally re-raise the error if you want the cell to fail, but after seeing the output
result.check_returncode()


Running dataset verification...

STDOUT:
[OK] train_c23.pickle: 1840 entries
  sample: ('original_sequences/youtube/c23/images/000/0000.png', 0)
[OK] val_c23.pickle: 220 entries
  sample: ('original_sequences/youtube/c23/images/001/0000.png', 0)
[OK] test_c23.pickle: 220 entries
  sample: ('original_sequences/youtube/c23/images/007/0000.png', 0)

STDERR:



In [ ]:
import torch

idx = torch.load(
    "/content/sfanet_data/processed/ffpp/train_c23.pickle",
    weights_only=False
)

print(idx[:20])

[('original_sequences/youtube/c23/images/000/0000.png', 0), ('original_sequences/youtube/c23/images/000/0001.png', 0), ('original_sequences/youtube/c23/images/000/0002.png', 0), ('original_sequences/youtube/c23/images/000/0003.png', 0), ('original_sequences/youtube/c23/images/000/0004.png', 0), ('original_sequences/youtube/c23/images/000/0005.png', 0), ('original_sequences/youtube/c23/images/000/0006.png', 0), ('original_sequences/youtube/c23/images/000/0007.png', 0), ('original_sequences/youtube/c23/images/000/0008.png', 0), ('original_sequences/youtube/c23/images/000/0009.png', 0), ('original_sequences/youtube/c23/images/002/0000.png', 0), ('original_sequences/youtube/c23/images/002/0001.png', 0), ('original_sequences/youtube/c23/images/002/0002.png', 0), ('original_sequences/youtube/c23/images/002/0003.png', 0), ('original_sequences/youtube/c23/images/002/0004.png', 0), ('original_sequences/youtube/c23/images/002/0005.png', 0), ('original_sequences/youtube/c23/images/002/0006.png', 

In [ ]:
import os

root = "/content/sfanet_data/processed/ffpp"

for split in ["train","val","test"]:
    print(split, os.path.exists(
        f"{root}/{split}_c23.pickle"
    ))

train True
val True
test True


In [ ]:
torch.load("/content/sfanet_data/processed/ffpp/train_c23.pickle")[:10]

[('original_sequences/youtube/c23/images/000/0000.png', 0),
 ('original_sequences/youtube/c23/images/000/0001.png', 0),
 ('original_sequences/youtube/c23/images/000/0002.png', 0),
 ('original_sequences/youtube/c23/images/000/0003.png', 0),
 ('original_sequences/youtube/c23/images/000/0004.png', 0),
 ('original_sequences/youtube/c23/images/000/0005.png', 0),
 ('original_sequences/youtube/c23/images/000/0006.png', 0),
 ('original_sequences/youtube/c23/images/000/0007.png', 0),
 ('original_sequences/youtube/c23/images/000/0008.png', 0),
 ('original_sequences/youtube/c23/images/000/0009.png', 0)]

In [ ]:
!find /content/sfanet_data/processed/ffpp -type d | head -30

/content/sfanet_data/processed/ffpp
/content/sfanet_data/processed/ffpp/manipulated_sequences
/content/sfanet_data/processed/ffpp/manipulated_sequences/Deepfakes
/content/sfanet_data/processed/ffpp/manipulated_sequences/Deepfakes/c23
/content/sfanet_data/processed/ffpp/manipulated_sequences/Deepfakes/c23/images
/content/sfanet_data/processed/ffpp/manipulated_sequences/Deepfakes/c23/images/092_098
/content/sfanet_data/processed/ffpp/manipulated_sequences/Deepfakes/c23/images/054_071
/content/sfanet_data/processed/ffpp/manipulated_sequences/Deepfakes/c23/images/059_050
/content/sfanet_data/processed/ffpp/manipulated_sequences/Deepfakes/c23/images/073_024
/content/sfanet_data/processed/ffpp/manipulated_sequences/Deepfakes/c23/images/027_009
/content/sfanet_data/processed/ffpp/manipulated_sequences/Deepfakes/c23/images/003_000
/content/sfanet_data/processed/ffpp/manipulated_sequences/Deepfakes/c23/images/071_054
/content/sfanet_data/processed/ffpp/manipulated_sequences/Deepfakes/c23/images

In [ ]:
import os

print(os.getcwd())

/content


In [ ]:
from dataset.faceforensics import FaceForensics
import sys

sys.path.append("/content/SFANet")

from dataset.faceforensics import FaceForensics

print("SUCCESS")

SUCCESS


In [ ]:
cfg = {
    "root": "/content/sfanet_data/processed/ffpp",
    "split": "train",
    "method": "all",
    "compression": "c23",
}

cfg = {
    "root": "/content/sfanet_data/processed/ffpp",
    "split": "train",
    "method": "all",
    "compression": "c23",
    "transforms": [],
}

dataset = FaceForensics(cfg)

print("Dataset size:", len(dataset))

Loading data from 'FF++ FaceSwap' of split 'train' and compression 'c23'
Please wait patiently...
Data from 'FF++' loaded.

Dataset contains 560 images.

Dataset size: 560


In [ ]:
sample_paths = dataset.images[:8]

imgs = dataset.load_item(sample_paths)

print(imgs.shape)

torch.Size([8, 3, 299, 299])


In [ ]:
sample_paths = dataset.images[:8]
imgs = dataset.load_item(sample_paths)
print(imgs.shape)

torch.Size([8, 3, 299, 299])


In [ ]:
# Stop Proceeding

In [ ]:
!apt-get install tree


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tree is already the newest version (2.0.2-1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [ ]:
!rm -rf sample_data


In [ ]:
!tree -L 2 SFANet

/bin/bash: line 1: tree: command not found


In [ ]:
!tree -L 4 /content/sfanet_data

In [ ]:
from pathlib import Path

ROOT = Path("/content/sfanet_data/raw/ffpp/FaceForensics++_C23")

print("ORIGINAL:")
for p in sorted(ROOT.joinpath("original").rglob("*.mp4"))[:20]:
    print(p.relative_to(ROOT))

ORIGINAL:
original/000.mp4
original/001.mp4
original/002.mp4
original/003.mp4
original/004.mp4
original/005.mp4
original/006.mp4
original/007.mp4
original/008.mp4
original/009.mp4
original/010.mp4
original/011.mp4
original/012.mp4
original/013.mp4
original/014.mp4
original/015.mp4
original/016.mp4
original/017.mp4
original/018.mp4
original/019.mp4


In [ ]:
print("\nDEEPFAKES:")
for p in sorted(ROOT.joinpath("Deepfakes").rglob("*.mp4"))[:20]:
    print(p.relative_to(ROOT))


DEEPFAKES:
Deepfakes/000_003.mp4
Deepfakes/001_870.mp4
Deepfakes/002_006.mp4
Deepfakes/003_000.mp4
Deepfakes/004_982.mp4
Deepfakes/005_010.mp4
Deepfakes/006_002.mp4
Deepfakes/007_132.mp4
Deepfakes/008_990.mp4
Deepfakes/009_027.mp4
Deepfakes/010_005.mp4
Deepfakes/011_805.mp4
Deepfakes/012_026.mp4
Deepfakes/013_883.mp4
Deepfakes/014_790.mp4
Deepfakes/015_919.mp4
Deepfakes/016_209.mp4
Deepfakes/017_803.mp4
Deepfakes/018_019.mp4
Deepfakes/019_018.mp4


In [ ]:
print("\nFACESWAP:")
for p in sorted(ROOT.joinpath("FaceSwap").rglob("*.mp4"))[:20]:
    print(p.relative_to(ROOT))


FACESWAP:
FaceSwap/000_003.mp4
FaceSwap/001_870.mp4
FaceSwap/002_006.mp4
FaceSwap/003_000.mp4
FaceSwap/004_982.mp4
FaceSwap/005_010.mp4
FaceSwap/006_002.mp4
FaceSwap/007_132.mp4
FaceSwap/008_990.mp4
FaceSwap/009_027.mp4
FaceSwap/010_005.mp4
FaceSwap/011_805.mp4
FaceSwap/012_026.mp4
FaceSwap/013_883.mp4
FaceSwap/014_790.mp4
FaceSwap/015_919.mp4
FaceSwap/016_209.mp4
FaceSwap/017_803.mp4
FaceSwap/018_019.mp4
FaceSwap/019_018.mp4


In [ ]:
print("\nCSV FILES:")
for p in sorted(ROOT.joinpath("csv").rglob("*"))[:50]:
    if p.is_file():
        print(p.relative_to(ROOT))


CSV FILES:
csv/DeepFakeDetection.csv
csv/Deepfakes.csv
csv/FF++_Metadata.csv
csv/FF++_Metadata_Shuffled.csv
csv/Face2Face.csv
csv/FaceShifter.csv
csv/FaceSwap.csv
csv/Mean_Data.csv
csv/NeuralTextures.csv
csv/original.csv


In [ ]:
print("\nCSV CONTENT SAMPLES:")

for p in sorted(ROOT.joinpath("csv").rglob("*.csv"))[:5]:
    print("\n---", p.relative_to(ROOT), "---")
    print(p.read_text(errors="ignore")[:1000])


CSV CONTENT SAMPLES:

--- csv/DeepFakeDetection.csv ---
,File Path,Label,Frame Count,Width,Height,Codec,File Size(MB)
0,DeepFakeDetection/01_02__meeting_serious__YVGY8LOK.mp4,FAKE,1044,1920,1080,h264,6.43
1,DeepFakeDetection/01_02__outside_talking_still_laughing__YVGY8LOK.mp4,FAKE,727,1920,1080,h264,5.05
2,DeepFakeDetection/01_02__talking_against_wall__YVGY8LOK.mp4,FAKE,841,1920,1080,h264,3.31
3,DeepFakeDetection/01_02__walk_down_hall_angry__YVGY8LOK.mp4,FAKE,217,1920,1080,h264,1.11
4,DeepFakeDetection/01_02__walking_down_indoor_hall_disgust__YVGY8LOK.mp4,FAKE,773,1920,1080,h264,12.05
5,DeepFakeDetection/01_03__hugging_happy__ISF9SP4G.mp4,FAKE,605,1920,1080,h264,8.7
6,DeepFakeDetection/01_03__kitchen_pan__JZUXXFRB.mp4,FAKE,455,1920,1080,h264,3.32
7,DeepFakeDetection/01_03__podium_speech_happy__480LQD1C.mp4,FAKE,898,1920,1080,h264,4.82
8,DeepFakeDetection/01_03__talking_against_wall__JZUXXFRB.mp4,FAKE,843,1920,1080,h264,3.33
9,DeepFakeDetection/01_04__hugging_happy__GBC7ZGDP.mp4,FAKE,5

In [ ]:
What we want instead

Suppose we have:

original/000.mp4
original/003.mp4

Deepfakes/000_003.mp4
Face2Face/000_003.mp4
FaceSwap/000_003.mp4
NeuralTextures/000_003.mp4

All of these should belong to one source group:

SOURCE GROUP {000, 003}
        │
        ├── original 000
        ├── original 003
        ├── Deepfakes 000_003
        ├── Face2Face 000_003
        ├── FaceSwap 000_003
        └── NeuralTextures 000_003

Then that entire group goes into exactly one of:

TRAIN
VAL
TEST

Never split it across them.

In [ ]:
from pathlib import Path
import re

ROOT = Path("/content/sfanet_data/raw/ffpp/FaceForensics++_C23")

methods = [
    "Deepfakes",
    "Face2Face",
    "FaceSwap",
    "NeuralTextures",
]

for method in methods:

    bad = []

    for p in (ROOT / method).rglob("*.mp4"):

        if not re.fullmatch(r"\d+_\d+", p.stem):
            bad.append(
                str(p.relative_to(ROOT))
            )

    print(
        f"{method}: "
        f"{len(bad)} unexpected filenames"
    )

    for x in bad[:5]:
        print("   ", x)

Deepfakes: 0 unexpected filenames
Face2Face: 0 unexpected filenames
FaceSwap: 0 unexpected filenames
NeuralTextures: 0 unexpected filenames


In [ ]:
from pathlib import Path
import pandas as pd

csv_path = "/content/sfanet_data/raw/ffpp/FaceForensics++_C23/csv/original.csv"

df = pd.read_csv(csv_path)

print(df.head())
print()
print("Rows:", len(df))

   Unnamed: 0         File Path Label  Frame Count  Width  Height Codec  \
0           0  original/000.mp4  REAL          396    640     480  h264   
1           1  original/001.mp4  REAL          460   1280     720  h264   
2           2  original/002.mp4  REAL          693   1280     720  h264   
3           3  original/003.mp4  REAL          303    640     480  h264   
4           4  original/004.mp4  REAL          309   1280     720  h264   

   File Size(MB)  
0           0.85  
1           2.78  
2           2.02  
3           0.58  
4           1.20  

Rows: 1000


In [ ]:
from pathlib import Path

root = Path("/content/sfanet_data/raw/ffpp/FaceForensics++_C23/original")

print("Number of original videos:",
      len(list(root.glob("*.mp4"))))

Number of original videos: 1000


In [ ]:
from pathlib import Path

root = Path("/content/sfanet_data/raw/ffpp/FaceForensics++_C23")

for p in root.rglob("*"):
    if "train" in p.name.lower() or \
       "test" in p.name.lower() or \
       "val" in p.name.lower():
        print(p)

In [ ]:
from pathlib import Path
import sys

sys.path.append("/content/SFANet/preprocessing")

from preprocessing_ffpp import _collect_video_jobs, FAKE_METHODS

raw_root = Path("/content/sfanet_data/raw/ffpp/FaceForensics++_C23")

jobs = _collect_video_jobs(
    raw_root,
    "c23",          # value doesn't matter much for inspection
    FAKE_METHODS,
    2               # only collect a few
)

print("Total jobs:", len(jobs))

for i in range(min(5, len(jobs))):
    print("\nJOB", i)
    print(jobs[i])

ModuleNotFoundError: No module named 'preprocess_ffpp'